# 02 Hypotheses and Evaluation

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['GE_DRIVE_ROOT'] = '/content/drive/MyDrive/hypothesaes_goemotions'
REPO_URL = 'https://github.com/YOUR_USERNAME/HypotheSAEs.git'   # <-- your fork
REPO_DIR = '/content/HypotheSAEs'

## Install

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-U', 'datasets', 'sentence-transformers',
                'transformers', 'scikit-learn', 'statsmodels'], check=True)
subprocess.run(['pip', 'install', '-q', 'vllm'], check=True)   # may force a Colab restart; rerun from here
# editable-install .pth files are only read at interpreter startup, so add the paths directly
import sys, importlib
for p in (REPO_DIR, os.path.join(REPO_DIR, 'goemotions')):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

## Imports

In [ ]:
import numpy as np
import pandas as pd
import ge_pipeline as ge
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.quickstart import train_sae
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import (NeuronInterpreter, InterpretConfig,
                                           SamplingConfig, LLMConfig, ScoringConfig)
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses
ge.set_seed()
ge.redirect_annotation_cache()

## Start Server

In [ ]:
import subprocess, time, requests
MODEL = 'Qwen/Qwen3-1.7B'
PORT = 8000
_server = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--port', str(PORT),
     '--max-model-len', '8192', '--gpu-memory-utilization', '0.85'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
BASE = f'http://127.0.0.1:{PORT}/v1'
for _ in range(180):
    try:
        if requests.get(BASE + '/models', timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError('vLLM server did not start')
os.environ['OPENAI_BASE_URL'] = BASE
os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')
print('server up at', BASE)

## Smoke Test

In [ ]:
from hypothesaes.llm_api import get_completion
print(get_completion(prompt='Reply with exactly: hello', model=MODEL,
                     max_output_tokens=2048, temperature=0.0))

## Load Data, Embeddings, SAE

In [ ]:
ds, NAMES = ge.load_goemotions()
train_texts = list(ds['train']['text'])
test_texts  = list(ds['test']['text'])
train_targets = ge.build_targets(ds['train']['labels'], NAMES)
test_targets  = ge.build_targets(ds['test']['labels'], NAMES)

t2e = get_local_embeddings(train_texts, model=ge.EMBEDDER, nomic_prefix=ge.NOMIC_PREFIX,
                           cache_name='goemotions_modernbert', batch_size=128)
train_emb = np.stack([t2e[t] for t in train_texts]).astype(np.float32)
del t2e; ge.clear_mem()

sae = train_sae(embeddings=train_emb, M=1024, K=8,
                checkpoint_dir=os.path.join(ge.CKPT_DIR, 'goemotions_M1024_K8'))
acts = sae.get_activations(train_emb)
del train_emb; ge.clear_mem()
acts.shape

## Select Neurons

In [ ]:
N_SELECT = 10
selected, sel_scores = {}, {}
for t in ge.TARGETS:
    if train_targets[t].sum() < 20:
        continue
    idxs, scores = select_neurons(activations=acts, target=train_targets[t],
                                  n_select=N_SELECT, method='correlation', classification=True)
    selected[t] = [int(i) for i in idxs]
    sel_scores[t] = [float(s) for s in scores]
union = sorted({i for v in selected.values() for i in v})
ge.log_json('05_selection', {'n_select': N_SELECT, 'selected': selected,
                             'scores': sel_scores, 'union_size': len(union)})
print('targets:', list(selected), '| union:', len(union))

## Interpret Neurons

In [ ]:
interpreter = NeuronInterpreter(interpreter_model=MODEL, annotator_model=MODEL,
                                n_workers_interpretation=8, n_workers_annotation=16,
                                cache_name='goemotions')
icfg = InterpretConfig(
    sampling=SamplingConfig(n_examples=20, max_words_per_example=64),
    llm=LLMConfig(temperature=0.7, max_output_tokens=4096),
    n_candidates=3,
    task_specific_instructions=ge.TASK_INSTRUCTIONS,
)
interps = interpreter.interpret_neurons(texts=train_texts, activations=acts,
                                        neuron_indices=union, config=icfg)

## Score Fidelity

In [ ]:
scfg = ScoringConfig(n_examples=100, max_words_per_example=64)
metrics = interpreter.score_interpretations(texts=train_texts, activations=acts,
                                            interpretations=interps, config=scfg, **ge.NO_THINK)

## Best Interpretations

In [ ]:
best = {}
for n in union:
    cands = [c for c in interps[n] if c]
    if not cands:
        best[n] = {'interpretation': None, 'f1': 0.0}
        continue
    bi = max(cands, key=lambda c: metrics[n][c]['f1'])
    best[n] = {'interpretation': bi, 'f1': float(metrics[n][bi]['f1'])}

neuron2hyp = {n: v['interpretation'] for n, v in best.items() if v['interpretation']}
ge.log_json('06_interpretations', {'best': {str(k): v for k, v in best.items()},
                                   'selected': selected})
del acts; ge.clear_mem()
pd.DataFrame([{'neuron': n, **v} for n, v in best.items()]).head(10)

## Annotate Heldout

In [ ]:
rng = np.random.default_rng(ge.SEED)
hidx = rng.choice(len(test_texts), size=min(ge.HELDOUT_CAP, len(test_texts)), replace=False)
held_texts = [test_texts[i] for i in hidx]
held_targets = {t: test_targets[t][hidx] for t in test_targets}

union_hyps = sorted(set(neuron2hyp.values()))
ann = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model=MODEL,
                                   cache_name='goemotions_heldout', n_workers=16,
                                   max_words_per_example=64, **ge.NO_THINK)

## Predictiveness

In [ ]:
results, rows = {}, []
for t, neurons in selected.items():
    hyps = sorted({neuron2hyp[n] for n in neurons if n in neuron2hyp})
    if len(hyps) < 2:
        continue
    m, df = score_hypotheses({h: ann[h] for h in hyps}, y_true=held_targets[t].astype(float),
                             classification=True, corrected_pval_threshold=0.1)
    results[t] = {'auroc': m.get('auroc'), 'r2': m.get('r2'),
                  'significant': m['Significant'][0], 'n_hyp': m['Significant'][1]}
    df['target'] = t
    rows.append(df)
table = pd.concat(rows, ignore_index=True)
ge.log_json('07_evaluation', {'per_target': results,
                              'heldout_indices': [int(i) for i in hidx],
                              'hypotheses': table.to_dict('records')})
pd.DataFrame(results).T

## SAE vs Interpreter

In [ ]:
sel_score = {(t, n): s for t in selected for n, s in zip(selected[t], sel_scores[t])}
recs = []
for t, neurons in selected.items():
    for n in neurons:
        h = neuron2hyp.get(n)
        if not h:
            continue
        sep = table[(table.target == t) & (table.hypothesis == h)]['separation_score']
        recs.append({'target': t, 'neuron': n, 'hypothesis': h,
                     'train_corr': abs(sel_score[(t, n)]),
                     'fidelity_f1': best[n]['f1'],
                     'heldout_sep': float(sep.iloc[0]) if len(sep) else None})
d8 = pd.DataFrame(recs)

def diagnose(r):
    if r['fidelity_f1'] < 0.5 and r['train_corr'] > 0.05:
        return 'interpreter_bottleneck'
    if r['train_corr'] < 0.03:
        return 'weak_feature'
    return 'ok'

d8['diagnosis'] = d8.apply(diagnose, axis=1)
ge.log_json('08_failure_localization', {'records': d8.to_dict('records'),
                                        'counts': d8['diagnosis'].value_counts().to_dict()})
print(d8['diagnosis'].value_counts().to_dict())
d8.sort_values('fidelity_f1').head(10)

## Worked Example

In [ ]:
d8[d8.diagnosis == 'interpreter_bottleneck'].sort_values('train_corr', ascending=False).head(3)

## Shutdown

In [ ]:
_server.terminate(); ge.clear_mem()